In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from astropy.table import Table

In [ ]:
%matplotlib inline

In [ ]:
MAG_ZPS_CPS = {
'u': 26.52,
'g': 28.51,
'r': 28.36,
'i': 28.17,
'z': 27.78,
'y': 26.82,
}

def flux2mag(flux, band, exposure_time=30.):
    output = -2.5 * np.log10(flux/exposure_time) + MAG_ZPS_CPS[band]
    return output


def mag2flux(mag, band, exposure_time=30.):
    flux = 10**((mag - MAG_ZPS_CPS[band]) / (-2.5)) * exposure_time
    return flux


In [ ]:
setid = 7
folder = "/home/sfu/shared_lagn_injection/lens_finding_postage_stamps_dataset"
filename = f"{folder}/3000sqdeg_lsst_1y_sample_{setid}.csv" 
csv = Table.read(filename)

In [ ]:
print("col0 min: ", np.min(csv['col0']) )
print("col0 max: ", np.max(csv['col0']) )

In [ ]:
missing_id = []
for i in range(np.max(csv['col0'])):
    if i not in csv['col0']:
        print("no exist id: ", i)
        missing_id.append(i)
print("missing_id: ", missing_id)

In [ ]:
def _correct_id(ind, missing_id=missing_id):
    count = 0
    if len(missing_id)==0:
        ind_new = ind
    elif len(missing_id)==1:
        if ind > missing_id[0]:
            ind_new = ind + 1
        else:
            ind_new
        
    return ind_new
    

def get_dx_new(ind, ind2, csv=csv):
    ind_new = _correct_id(ind)
    sel = csv['col0']==ind_new
    row = csv[sel][0]
    tmp = row[f'point_source_light_i_ra_image_{ind2}'] / 0.2
    return tmp

def get_dy_new(ind, ind2):
    ind_new = _correct_id(ind)
    sel = csv['col0']==ind_new
    row = csv[sel][0]
    tmp = row[f'point_source_light_i_dec_image_{ind2}'] / 0.2
    return tmp


In [ ]:
#hf["lsst_lens_0"].visit(print)

In [ ]:
#filename = "/home/sfu/shared_lagn_injection/3000sqdeg_lsst_1y_sample_merged_cleaned.h5"

filename = f"{folder}/3000sqdeg_lsst_1y_sample_{setid}.h5" 

hf = h5py.File(filename, "r")

print('num of rows: ', len(hf) )

In [ ]:
maxid = 0
for key in hf.keys():
    tmp = int(key.split('_')[-1])
    if tmp > maxid:
        maxid = tmp
print("max id:", maxid)

In [ ]:
for i in range(maxid+1):
    if f'lsst_lens_{i}' not in hf.keys():
        print("no exist id: ", i)

In [ ]:
def get_dx(ind, ind2, hf=hf):
    return hf[f'lsst_lens_{ind}']['metadata'].attrs[f'point_source_light_i_ra_image_{ind2}'] / 0.2

def get_dy(ind, ind2, hf=hf):
    return hf[f'lsst_lens_{ind}']['metadata'].attrs[f'point_source_light_i_dec_image_{ind2}'] / 0.2

In [ ]:
def plot_static(ind, band, hf=hf):
    arr = hf[f'lsst_lens_{ind}']['static_image'][band]['lens_plus_lensed_agn_host'][:]
    plt.figure(figsize=(4,4))
    plt.imshow(np.arcsinh(arr), origin="lower")
    #plt.imshow(arr, origin="lower")
    plt.colorbar()

def _plot_point(ind):
    x_c, y_c = 16, 16
    #for ind2 in [0,1]:
    for ind2 in [1,0]:
        #dx = get_dx(ind, ind2)
        #dy = get_dy(ind, ind2)
        dx = get_dx_new(ind, ind2)
        dy = get_dy_new(ind, ind2)
        x = x_c + dx
        y = y_c + dy
        plt.plot([x], [y], 'm.')
    return int(x), int(y)
    

def plot_stp(ind, band, time_index=0, hf=hf, if_point=True):
    arr = hf[f'lsst_lens_{ind}']['postage_stamps'][band]['all_observations'][time_index]
    plt.figure(figsize=(4,4))
    plt.imshow(np.arcsinh(arr), origin="lower")
    #plt.imshow(arr, origin="lower")
    plt.colorbar()

    if if_point:
        _plot_point(ind)

def plot_diff(ind, band, time_index=0, hf=hf, if_point=True):
    arr_static = hf[f'lsst_lens_{ind}']['static_image'][band]['lens_plus_lensed_agn_host'][:]
    arr_stamp = hf[f'lsst_lens_{ind}']['postage_stamps'][band]['all_observations'][time_index]
    arr_stamp_mean = np.mean(hf[f'lsst_lens_{ind}']['postage_stamps'][band]['all_observations'][:], axis=0)
    
    plt.figure(figsize=(4,4))
    
    #plt.imshow(arr_stamp - arr_stamp_mean, origin="lower", vmin=-20, vmax=20, cmap='bwr')
    #plt.imshow(arr_stamp - arr_static, origin="lower")
    #plt.imshow(np.arcsinh(arr_stamp - arr_stamp_mean), origin="lower", vmin=-2, vmax=2, cmap='bwr')
    plt.imshow(np.arcsinh(gaussian_filter(arr_stamp - arr_stamp_mean, sigma=2)), 
               origin="lower", vmin=-8, vmax=8, cmap='bwr')
    
    #plt.imshow(np.log10(arr_stamp - arr_stamp_mean), origin="lower")
    
    plt.colorbar()
    #plt.title('stamp - stamp_mean')

    if if_point:
        x, y = _plot_point(ind)
        point_src = arr_stamp - arr_static
        arr_sel = point_src[y-2:y+3, x-2:x+3]
        print(arr_sel)
        total_flux = np.sum(arr_sel)
        light_curves = hf[f'lsst_lens_{ind}']['light_curves']
        #image_ind = 1
        image_ind = 0
        mag = light_curves[f'image_{image_ind}'][band][time_index]
        print("flux_mag, lc_mag:", flux2mag(total_flux, band), mag)
        

In [ ]:
ind = 233
band = 'i'

In [ ]:
hf[f'lsst_lens_{ind}']['light_curves']

In [ ]:
plot_static(ind, 'i')

In [ ]:
plot_stp(ind, 'i')

In [ ]:
plot_diff(ind, 'i', 0)

In [ ]:
plot_diff(ind, 'i', 2)

In [ ]:
hf.close()